# Day 13 - Data Cleaning

## Week 2 topic

- **Topic:** Data cleaning
- **Summary:** Preparing data for ML algorithms, missing values, `SimpleImputer`, and separating labels from features
- **Practice:** Build an imputation step and verify there are no missing values

### Learning goals

By the end of this lesson, you should be able to:

1. Distinguish predictors (features) from the value to predict (the label).
2. Detect missing values before sending data to an ML algorithm.
3. Explain why an imputer must be fit on training data only.
4. Build a reusable `SimpleImputer` transformation.
5. Verify that transformed feature matrices contain no missing values.

### Connection to the book

This lesson follows the data-preparation ideas in *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2nd edition), especially the California Housing workflow in Chapter 2. The important rule is: learn cleaning statistics from the training set, then apply those learned statistics to validation or test data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 1. Load and Inspect the Dataset

Before cleaning, inspect the shape, columns, data types, and missing-value counts. The local CSV from Day 9 contains the predictor columns but not the target column, so this cell uses the complete scikit-learn frame when the target is absent. This keeps the lesson runnable while still pointing to the project data directory first.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
CSV_CANDIDATES = [
    PROJECT_ROOT / "day9_Get_and_Inspect_Data" / "california_housing.csv",
    PROJECT_ROOT.parent / "day9_Get_and_Inspect_Data" / "california_housing.csv",
]

csv_path = next((path for path in CSV_CANDIDATES if path.exists()), None)
candidate_df = pd.read_csv(csv_path) if csv_path else None

if candidate_df is not None and "MedHouseVal" in candidate_df.columns:
    df = candidate_df
    data_source = f"CSV: {csv_path}"
else:
    housing = fetch_california_housing(as_frame=True)
    df = housing.frame.copy()
    data_source = "scikit-learn California Housing frame"

print("Data source:", data_source)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Data source: scikit-learn California Housing frame
Shape: (20640, 9)
Columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
print("Data types:")
df.info()

print("\nMissing values before the practice setup:")
print(df.isna().sum())

print("\nSummary statistics:")
df.describe().T

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB

Missing values before the practice setup:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Summary statistics:


,count,mean,std,min,25%,50%,75%,max
MedInc,20640.0,3.870671,1.899822,0.499900,2.563400,3.534800,4.743250,15.000100
HouseAge,20640.0,28.639486,12.585558,1.000000,18.000000,29.000000,37.000000,52.000000
AveRooms,20640.0,5.429000,2.474173,0.846154,4.440716,5.229129,6.052381,141.909091
AveBedrms,20640.0,1.096675,0.473911,0.333333,1.006079,1.048780,1.099526,34.066667
Population,20640.0,1425.476744,1132.462122,3.000000,787.000000,1166.000000,1725.000000,35682.000000
AveOccup,20640.0,3.070655,10.386050,0.692308,2.429741,2.818116,3.282261,1243.333333
Latitude,20640.0,35.631861,2.135952,32.540000,33.930000,34.260000,37.710000,41.950000
Longitude,20640.0,-119.569704,2.003532,-124.350000,-121.800000,-118.490000,-118.010000,-114.310000
MedHouseVal,20640.0,2.068558,1.153956,0.149990,1.196000,1.797000,2.647250,5.000010


## 2. Separate Labels from Features

The **label** (`y`) is the value the model should predict. The **features** (`X`) are the input columns used to make that prediction. Keeping them separate prevents us from accidentally imputing or scaling the target as if it were an input.

For this lesson, `MedHouseVal` is the label and all remaining numeric columns are features.

In [4]:
TARGET = "MedHouseVal"
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Feature shape:", X.shape)
print("Label shape:", y.shape)
print("Feature columns:", list(X.columns))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Feature shape: (20640, 8)
Label shape: (20640,)
Feature columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Training rows: 16512
Test rows: 4128


## 3. Create Missing Values for Practice

The downloaded California Housing data is complete, so there may be nothing for `SimpleImputer` to repair. To make the practice observable, we copy the train and test features and replace a small, reproducible sample of values with `NaN`.

In a real project, these values would already be missing in the source data. The lesson is the same: inspect missingness, learn replacement values from training data, and apply the transformation consistently.

In [5]:
X_train_dirty = X_train.copy()
X_test_dirty = X_test.copy()

# Fixed row/column positions make this experiment reproducible.
missing_train_rows = X_train_dirty.index[:12]
missing_test_rows = X_test_dirty.index[:5]
X_train_dirty.loc[missing_train_rows, "AveRooms"] = np.nan
X_train_dirty.loc[missing_train_rows[:6], "Population"] = np.nan
X_test_dirty.loc[missing_test_rows, "AveRooms"] = np.nan
X_test_dirty.loc[missing_test_rows[:3], "Population"] = np.nan

print("Missing values in dirty training features:")
print(X_train_dirty.isna().sum())
print("\nMissing values in dirty test features:")
print(X_test_dirty.isna().sum())

Missing values in dirty training features:
MedInc         0
HouseAge       0
AveRooms      12
AveBedrms      0
Population     6
AveOccup       0
Latitude       0
Longitude      0
dtype: int64

Missing values in dirty test features:
MedInc        0
HouseAge      0
AveRooms      5
AveBedrms     0
Population    3
AveOccup      0
Latitude      0
Longitude     0
dtype: int64


## 4. Build a `SimpleImputer` Step

`SimpleImputer(strategy="median")` replaces each missing numeric value with the median of its column. Median is often a practical choice for skewed numeric data because it is less affected by extreme values than the mean.

The crucial order is:

1. Call `fit` on `X_train_dirty` only, so the medians come from training data.
2. Call `transform` on both train and test features using those same medians.
3. Never calculate replacement values from the test set.

In [6]:
imputer = SimpleImputer(strategy="median")
X_train_imputed_array = imputer.fit_transform(X_train_dirty)
X_test_imputed_array = imputer.transform(X_test_dirty)

X_train_imputed = pd.DataFrame(
    X_train_imputed_array, columns=X_train.columns, index=X_train.index
)
X_test_imputed = pd.DataFrame(
    X_test_imputed_array, columns=X_test.columns, index=X_test.index
)

print("Learned medians:")
print(pd.Series(imputer.statistics_, index=X_train.columns))
print("\nImputed training sample:")
display(X_train_imputed.loc[missing_train_rows, ["AveRooms", "Population"]])

Learned medians:
MedInc           3.545800
HouseAge        29.000000
AveRooms         5.235874
AveBedrms        1.049286
Population    1167.000000
AveOccup         2.817240
Latitude        34.260000
Longitude     -118.510000
dtype: float64

Imputed training sample:


,AveRooms,Population
14196,5.235874,1167.0
8267,5.235874,1167.0
17445,5.235874,1167.0
14265,5.235874,1167.0
2271,5.235874,1167.0
17848,5.235874,1167.0
6252,5.235874,1355.0
9389,5.235874,999.0
6113,5.235874,819.0
6061,5.235874,9427.0


## 5. Put the Cleaning Step in a Pipeline

A pipeline keeps preprocessing steps together and makes the same transformation easy to reuse later with a model. For this lesson, the pipeline contains one step: median imputation. In a larger project, it could also contain scaling, encoding, and an estimator.

In [7]:
numeric_preprocessor = Pipeline(
    steps=[("median_imputer", SimpleImputer(strategy="median"))]
)

X_train_clean = numeric_preprocessor.fit_transform(X_train_dirty)
X_test_clean = numeric_preprocessor.transform(X_test_dirty)

print("Clean training matrix shape:", X_train_clean.shape)
print("Clean test matrix shape:", X_test_clean.shape)

Clean training matrix shape: (16512, 8)
Clean test matrix shape: (4128, 8)


## 6. Verify the Practice Result

The assertions below are the completion check for this lesson. They verify that the imputer removed missing values from both transformed matrices, that the label remains separate, and that the feature count did not change.

In [8]:
assert np.isnan(X_train_clean).sum() == 0
assert np.isnan(X_test_clean).sum() == 0
assert TARGET not in X.columns
assert X_train_clean.shape[1] == X_train.shape[1]
assert X_test_clean.shape[1] == X_test.shape[1]

print("Missing values after imputation - train:", np.isnan(X_train_clean).sum())
print("Missing values after imputation - test:", np.isnan(X_test_clean).sum())
print("Labels remain separate:", TARGET not in X.columns)
print("Practice check passed: the imputed feature matrices contain no missing values.")

Missing values after imputation - train: 0
Missing values after imputation - test: 0
Labels remain separate: True
Practice check passed: the imputed feature matrices contain no missing values.


## 7. Review and Extend

### Key takeaways

- Missing values must be handled before most scikit-learn estimators can train.
- `SimpleImputer` learns a replacement value per feature.
- Fit preprocessing on training data only to avoid data leakage.
- Keep labels separate from the feature matrix.
- A pipeline makes preprocessing repeatable and ready to connect to a model.

### Practice extensions

1. Replace the median strategy with `mean` and compare the learned statistics.
2. Try `strategy="most_frequent"` on a small categorical example.
3. Add a model after the preprocessing pipeline and evaluate it on `X_test_clean` and `y_test`.
4. Compare the result with a version that drops incomplete rows. Which approach preserves more data?